# EXP-007: Ridge Meta-Learner Stack + 0.30/0.70 Blend with PF Heuristic

In [1]:
import sys
import numpy as np
import pandas as pd
import lightgbm as lgb
import catboost as cb
import json
import time
import joblib
import warnings
import threading
from joblib import Parallel, delayed
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

# -- Config --------------------------------------------------------
EXP_ID      = 'EXP-007'
TINY_SAMPLE = False   # True = first 10 wells only (sanity check)
N_FOLDS     = 5
SEED        = 42
BLEND_ALPHA = 0.30    # Ridge weight; 0.70 on heuristic

# -- Paths ---------------------------------------------------------
IS_KAGGLE = Path('/kaggle/input').exists()

# -- GPU Detection -------------------------------------------------
try:
    import torch
    n_gpus = torch.cuda.device_count()
except Exception:
    try:
        import catboost as cb
        n_gpus = cb.utils.get_gpu_device_count()
    except Exception:
        n_gpus = 0
if n_gpus < 1:
    n_gpus = 1
USE_GPU = n_gpus > 0

# Input: output of rogii-eda-features.ipynb (train_df.joblib + features.json)
PREP_CANDIDATES = [
    Path('/kaggle/input/notebooks/baopv05/rogii-eda-features'),
    Path('./output_aw'),
]
PREP_DIR = next((p for p in PREP_CANDIDATES if (p / 'train_df.joblib').exists()),
                PREP_CANDIDATES[-1])

# Input: pre-trained OOFs from rogii-train-lightgbm / rogii-train-catboost
LGB_OOF_CANDIDATES = [
    Path('/kaggle/input/notebooks/baopv051/rogii-train-lightgbm'),
    Path('./output_aw_lgbm'),
]
CB_OOF_CANDIDATES = [
    Path('/kaggle/input/notebooks/baopv051/rogii-train-catboost'),
    Path('./output_aw_catboost'),
]
LGB_DIR = next((p for p in LGB_OOF_CANDIDATES if (p / 'oof').exists()), None)
CB_DIR  = next((p for p in CB_OOF_CANDIDATES  if (p / 'oof').exists()), None)

# Input: pre-built test features (test_df.joblib produced by rogii-eda-features)
TEST_DF_CANDIDATES = [
    Path('/kaggle/input/notebooks/baopv05/rogii-eda-features'),
    Path('./output_aw'),
    Path('/kaggle/working'),
]

# Output
WORK_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('./output_exp007')
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f'EXP_ID      = {EXP_ID}')
print(f'TINY_SAMPLE = {TINY_SAMPLE}')
print(f'BLEND_ALPHA = {BLEND_ALPHA} Ridge / {1-BLEND_ALPHA} heuristic')
print(f'PREP_DIR    = {PREP_DIR}')
print(f'LGB_DIR     = {LGB_DIR}')
print(f'CB_DIR      = {CB_DIR}')
print(f'GPUs detected: {n_gpus} (USE_GPU={USE_GPU})')

EXP_ID      = EXP-007
TINY_SAMPLE = False
BLEND_ALPHA = 0.3 Ridge / 0.7 heuristic
PREP_DIR    = /kaggle/input/notebooks/baopv05/rogii-eda-features
LGB_DIR     = None
CB_DIR      = None
GPUs detected: 2 (USE_GPU=True)


In [2]:
# -- Load train_df from rogii-eda-features pipeline ----------------
# train_df is eval-zone-only rows; target = residual (TVT - last_known_tvt)
train_df_full = joblib.load(PREP_DIR / 'train_df.joblib')
with open(PREP_DIR / 'features.json', encoding='utf-8') as f:
    FEATURE_COLS = json.load(f)

N_FULL = len(train_df_full)
print(f'Loaded train_df: {N_FULL} rows, {len(FEATURE_COLS)} features')

# Build TINY_SAMPLE mask (applied to pre-trained OOFs later too)
if TINY_SAMPLE:
    sample_wells = np.unique(train_df_full['well'].values)[:10]
    tiny_mask = np.isin(train_df_full['well'].values, sample_wells)
    print(f'TINY_SAMPLE: {len(sample_wells)} wells, {tiny_mask.sum()} eval rows')
else:
    tiny_mask = None  # No filtering

train_df    = train_df_full[tiny_mask].reset_index(drop=True) if tiny_mask is not None else train_df_full
X_eval      = train_df[FEATURE_COLS]
y_eval      = train_df['target'].values.astype(np.float32)
groups_eval = train_df['well'].values

print(f'Train (eval-zone): {X_eval.shape}  Target (residual): {y_eval.shape}')
print(f'Wells: {len(np.unique(groups_eval))}')
print(f'Target: mean={y_eval.mean():.2f}  std={y_eval.std():.2f}')

# Check train/test well overlap
overlap = set(train_df['well'].unique())
print(f'Train wells: {len(overlap)}  (EXP-011 overlap check will compare with test wells)')

Loaded train_df: 3783989 rows, 222 features
Train (eval-zone): (3783989, 222)  Target (residual): (3783989,)
Wells: 773
Target: mean=1.60  std=15.83
Train wells: 773  (EXP-011 overlap check will compare with test wells)


In [3]:
# -- Load test features (test_df.joblib) ---------------------------
# Expects test_df.joblib produced by rogii-eda-features.ipynb (with test mode enabled)
# OR by the feature-building section of rogii-submission.ipynb.
X_test   = None
test_df  = None
HAS_TEST = False

for p in TEST_DF_CANDIDATES:
    test_joblib = p / 'test_df.joblib'
    if test_joblib.exists():
        test_df  = joblib.load(test_joblib)
        # Align feature columns (test may have been built with same feature set)
        test_feat_cols = [c for c in FEATURE_COLS if c in test_df.columns]
        X_test   = test_df[FEATURE_COLS].fillna(0).values.astype(np.float32)
        HAS_TEST = True
        print(f'Loaded pre-built test features: {test_joblib}')
        print(f'  test_df: {test_df.shape}  X_test: {X_test.shape}')
        break

if not HAS_TEST:
    print('WARNING: test_df.joblib not found in any candidate path.')
    print('To produce test_df.joblib, run rogii-eda-features.ipynb with test mode enabled')
    print('(extend build_dataset to also process DATA_DIR/test/ and save test_df.joblib).')
    print('Proceeding in OOF-only mode â€” test_matrix will be zeros.')
    X_test = np.zeros((0, X_eval.shape[1]), dtype=np.float32)

if HAS_TEST and TINY_SAMPLE and test_df is not None:
    sample_wells_set = set(np.unique(groups_eval))
    test_mask = np.isin(test_df['well'].values, list(sample_wells_set))
    X_test  = X_test[test_mask]
    test_df = test_df[test_mask].reset_index(drop=True)
    print(f'TINY_SAMPLE test: {X_test.shape}')

print(f'HAS_TEST = {HAS_TEST}  X_test.shape = {X_test.shape}')

To produce test_df.joblib, run rogii-eda-features.ipynb with test mode enabled
(extend build_dataset to also process DATA_DIR/test/ and save test_df.joblib).
Proceeding in OOF-only mode â€” test_matrix will be zeros.
HAS_TEST = False  X_test.shape = (0, 222)


In [4]:
# -- Try loading pre-trained OOFs from rogii-train-lightgbm / catboost --
# OOFs have shape (N_FULL,) in residual space â€” same as y_eval
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

loaded_oofs    = {}   # label â†’ (N_eval,) residual OOF array
loaded_models  = {}   # label â†’ list of fold models (for test prediction)

def _try_load_oof(model_dir, model_type, idx):
    if model_dir is None:
        return
    name = f'lightgbm-{idx}' if model_type == 'LGB' else f'catboost-{idx}'
    label = f'{model_type}-{idx}'
    npy = model_dir / 'oof' / f'{name}_oof.npy'
    if not npy.exists():
        return
    arr = np.load(npy).astype(np.float32)
    if arr.shape[0] != N_FULL:
        print(f'  Shape mismatch {label}: {arr.shape[0]} vs {N_FULL} â€” will retrain')
        return
    arr_sub = arr[tiny_mask] if tiny_mask is not None else arr
    loaded_oofs[label] = arr_sub
    # Also try loading fold models for test prediction
    mfile = model_dir / 'models' / f'{name}.joblib'
    if mfile.exists():
        loaded_models[label] = joblib.load(mfile)
    cv = rmse(y_eval, arr_sub)
    print(f'  Loaded OOF {label}: CV={cv:.5f}  n_models={len(loaded_models.get(label, []))} fold models')

for i in range(1, 4):
    _try_load_oof(LGB_DIR, 'LGB', i)
    _try_load_oof(CB_DIR,  'CB',  i)

print(f'Pre-loaded OOFs: {list(loaded_oofs.keys())}')

# -- Base model configs (Dynamic GPU/CPU) --------------------------
LGB_BASE = {'objective': 'regression', 'metric': 'rmse', 'verbosity': -1,
            'n_jobs': -1, 'seed': SEED, 'device': 'gpu' if USE_GPU else 'cpu'}
LGB_CONFIGS = [
    # LGB-1: aw pipeline baseline
    {**LGB_BASE, 'num_leaves': 255, 'learning_rate': 0.02,    'reg_lambda': 3.0,
     'n_estimators': 5000,  'subsample': 0.8,   'colsample_bytree': 0.8},
    # LGB-2: medium complexity
    {**LGB_BASE, 'num_leaves': 127, 'learning_rate': 0.015,   'reg_lambda': 10.0,
     'n_estimators': 7000,  'subsample': 0.7,   'colsample_bytree': 0.7},
    # LGB-3: Optuna heavy-regularization (rogii-sel15, H-011)
    {**LGB_BASE, 'num_leaves': 64,  'learning_rate': 0.00935, 'reg_alpha': 10.79,
     'reg_lambda': 95.75, 'colsample_bytree': 0.393, 'min_child_samples': 40,
     'subsample': 0.474, 'n_estimators': 10000},
]

CB_BASE = {'loss_function': 'RMSE', 'eval_metric': 'RMSE', 'verbose': 0,
           'random_seed': SEED, 'task_type': 'GPU' if USE_GPU else 'CPU',
           'bootstrap_type': 'Bernoulli'}
CB_CONFIGS = [
    # CB-1: catboost-2 from aw blend (weight 0.272)
    {**CB_BASE, 'iterations': 5000, 'learning_rate': 0.03, 'depth': 8,
     'l2_leaf_reg': 3.0, 'subsample': 0.8},
    # CB-2: catboost-3 from aw blend (weight 0.376)
    {**CB_BASE, 'iterations': 5000, 'learning_rate': 0.02, 'depth': 10,
     'l2_leaf_reg': 5.0, 'subsample': 0.75},
    # CB-3: lower lr, deeper
    {**CB_BASE, 'iterations': 8000, 'learning_rate': 0.01, 'depth': 10,
     'l2_leaf_reg': 8.0, 'subsample': 0.7},
]
print(f'{len(LGB_CONFIGS)} LGB + {len(CB_CONFIGS)} CB = {len(LGB_CONFIGS)+len(CB_CONFIGS)} base models')

Pre-loaded OOFs: []
3 LGB + 3 CB = 6 base models


In [5]:
# -- Train base models â†’ collect OOF matrix ------------------------
kf      = GroupKFold(n_splits=N_FOLDS)
splits  = list(kf.split(X_eval, groups=groups_eval))
n_models    = len(LGB_CONFIGS) + len(CB_CONFIGS)
oof_matrix  = np.zeros((len(X_eval), n_models), dtype=np.float32)
test_matrix = np.zeros((X_test.shape[0], n_models), dtype=np.float32) if HAS_TEST else np.zeros((0, n_models), dtype=np.float32)
model_scores = []
t_total = time.time()

for m_idx, (model_type, configs) in enumerate([('LGB', LGB_CONFIGS), ('CB', CB_CONFIGS)]):
    for cfg_idx, cfg in enumerate(configs):
        col   = (3 * (model_type == 'CB')) + cfg_idx
        label = f'{model_type}-{cfg_idx+1}'

        # -- Fast path: use pre-loaded OOF -----------------â”        # -- Slow path: train from scratch in parallel --------------
        print(f'\n-- {label} (training from scratch in parallel) --')
        t0 = time.time()
        oof_fold  = np.zeros(len(X_eval), dtype=np.float32)
        test_fold = np.zeros(X_test.shape[0], dtype=np.float32) if HAS_TEST else np.array([])

        # To avoid GPU collision, we assign a GPU ID to each thread
        gpu_lock = threading.Lock()
        available_gpus = list(range(n_gpus)) if USE_GPU else []

        def train_one_fold(fold, tr_idx, val_idx):
            gpu_id = None
            if USE_GPU:
                with gpu_lock:
                    if available_gpus:
                        gpu_id = available_gpus.pop(0)
                    else:
                        gpu_id = 0
            
            fold_cfg = cfg.copy()
            if USE_GPU and gpu_id is not None:
                if model_type == 'LGB':
                    fold_cfg['gpu_device_id'] = gpu_id
                else:
                    fold_cfg['devices'] = str(gpu_id)
            
            try:
                X_tr, X_val = X_eval.iloc[tr_idx], X_eval.iloc[val_idx]
                y_tr, y_val = y_eval[tr_idx], y_eval[val_idx]
                
                if model_type == 'LGB':
                    model = lgb.LGBMRegressor(**fold_cfg)
                    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                              callbacks=[lgb.early_stopping(250, verbose=False), lgb.log_evaluation(0)])
                else:
                    model = cb.CatBoostRegressor(**fold_cfg)
                    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=300)
                
                val_preds = model.predict(X_val)
                test_preds = model.predict(X_test) if HAS_TEST else None
                
                fold_rmse = rmse(y_val, val_preds)
                gpu_msg = f' on GPU {gpu_id}' if gpu_id is not None else ''
                print(f'  fold {fold+1}{gpu_msg}: RMSE={fold_rmse:.5f}  {time.time()-t0:.0f}s')
                return fold, val_idx, val_preds, test_preds, model
            finally:
                if USE_GPU and gpu_id is not None:
                    with gpu_lock:
                        available_gpus.append(gpu_id)

        # Run folds in parallel
        n_workers = min(n_gpus, N_FOLDS)
        results = Parallel(n_jobs=n_workers, backend='threading')(
            delayed(train_one_fold)(fold, tr_idx, val_idx)
            for fold, (tr_idx, val_idx) in enumerate(splits)
        )
        
        # Aggregate results
        fold_models = [None] * N_FOLDS
        for fold, val_idx, val_preds, test_preds, model in results:
            oof_fold[val_idx] = val_preds
            fold_models[fold] = model
            if HAS_TEST and test_preds is not None:
                test_fold += test_preds / N_FOLDS

        oof_matrix[:, col]  = oof_fold
        if HAS_TEST:
            test_matrix[:, col] = test_fold
        cv = rmse(y_eval, oof_fold)
        model_scores.append({'model': label, 'cv_rmse': round(cv, 6)})
        print(f'  {label} CV={cv:.5f}')
        np.save(WORK_DIR / f'{EXP_ID}_{label}_oof.npy',  oof_fold)
        if HAS_TEST:
            np.save(WORK_DIR / f'{EXP_ID}_{label}_test.npy', test_fold)
        
        # Save retrained fold models
        file_name = f'lightgbm-{cfg_idx+1}' if model_type == 'LGB' else f'catboost-{cfg_idx+1}'
        joblib.dump(fold_models, WORK_DIR / f'{EXP_ID}_{file_name}_models.joblib')

np.save(WORK_DIR / f'{EXP_ID}_oof_matrix.npy',  oof_matrix)
if HAS_TEST:
    np.save(WORK_DIR / f'{EXP_ID}_test_matrix.npy', test_matrix)
print(f'\nAll base models: {(time.time()-t_total)/60:.1f} min')
for s in model_scores:
    print(f'  {s["model"]}: {s["cv_rmse"]}')


-- LGB-1 (training from scratch in parallel) --


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning gen

  fold 1 on GPU 0: RMSE=9.62951  430s
  fold 3 on GPU 0: RMSE=9.77940  1033s
  fold 4 on GPU 0: RMSE=11.53471  1419s
  fold 5 on GPU 0: RMSE=11.07986  2071s
  fold 2 on GPU 1: RMSE=11.39645  2111s
  LGB-1 CV=10.71486

-- LGB-2 (training from scratch in parallel) --
  fold 1 on GPU 0: RMSE=9.55801  509s
  fold 3 on GPU 0: RMSE=9.66820  1125s
  fold 4 on GPU 0: RMSE=11.48274  1390s
  fold 2 on GPU 1: RMSE=11.23152  1992s
  fold 5 on GPU 0: RMSE=11.06016  2121s
  LGB-2 CV=10.63149

-- LGB-3 (training from scratch in parallel) --
  fold 2 on GPU 1: RMSE=11.13143  870s
  fold 1 on GPU 0: RMSE=9.49015  962s
  fold 4 on GPU 0: RMSE=11.39402  1350s
  fold 3 on GPU 1: RMSE=9.70455  2064s
  fold 5 on GPU 0: RMSE=11.12151  2339s
  LGB-3 CV=10.59852

-- CB-1 (training from scratch in parallel) --
  fold 1 on GPU 0: RMSE=9.81673  132s
  fold 3 on GPU 0: RMSE=9.61489  268s
  fold 4 on GPU 0: RMSE=11.52089  316s
  fold 2 on GPU 1: RMSE=11.14655  342s
  fold 5 on GPU 0: RMSE=11.10842  665s
  CB-1 CV=1

In [6]:
# ── Ridge meta-learner (inline — no external import needed) ───────
class RidgeStacker:
    def __init__(self, alpha=1.0, blend_alpha=0.30, seed=42):
        self.alpha = alpha; self.blend_alpha = blend_alpha; self.seed = seed
        self.scaler = StandardScaler(); self.ridge = Ridge(alpha=alpha)
    def fit(self, X, y):
        self.ridge.fit(self.scaler.fit_transform(X), y); return self
    def predict_stack(self, X):
        return self.ridge.predict(self.scaler.transform(X))
    def blend(self, stack, heuristic):
        return self.blend_alpha * stack + (1 - self.blend_alpha) * heuristic
    def fit_predict_oof(self, X, y, cv=5, groups=None):
        oof = np.zeros(len(y))
        kf_i = GroupKFold(cv) if groups is not None else __import__('sklearn.model_selection', fromlist=['KFold']).KFold(cv, shuffle=True, random_state=self.seed)
        splits_ = list(kf_i.split(X, groups=groups) if groups is not None else kf_i.split(X))
        for tr, val in splits_:
            sc = StandardScaler()
            r  = Ridge(alpha=self.alpha)
            r.fit(sc.fit_transform(X[tr]), y[tr])
            oof[val] = r.predict(sc.transform(X[val]))
        self.fit(X, y)
        return oof

stacker   = RidgeStacker(alpha=1.0, blend_alpha=BLEND_ALPHA, seed=SEED)
oof_stack = stacker.fit_predict_oof(oof_matrix, y_eval, cv=N_FOLDS, groups=groups_eval)
print(f'Ridge stack OOF RMSE: {rmse(y_eval, oof_stack):.5f}')
print(f'Weights: {dict(zip([s["model"] for s in model_scores], stacker.ridge.coef_.round(4)))}')
np.save(WORK_DIR / f'{EXP_ID}_ridge_stack_oof.npy', oof_stack)
if HAS_TEST:
    test_stack = stacker.predict_stack(test_matrix)
    np.save(WORK_DIR / f'{EXP_ID}_ridge_stack_test.npy', test_stack)
else:
    test_stack = None
    print('HAS_TEST=False: test_stack skipped')

Ridge stack OOF RMSE: 10.56195
Weights: {'LGB-1': np.float32(0.8384), 'LGB-2': np.float32(2.5429), 'LGB-3': np.float32(3.0629), 'CB-1': np.float32(0.6644), 'CB-2': np.float32(1.2424), 'CB-3': np.float32(3.6308)}
HAS_TEST=False: test_stack skipped


In [7]:
# -- PF heuristic from train_df (pf_ancc / pf_z columns) ----------
# train_df['pf_ancc'] is absolute TVT â†’ convert to residual space
pf_oof_resid = (train_df['pf_ancc'].values - train_df['last_known_tvt'].values).astype(np.float32)

# Blend ANCC and Z-based particle filter if both available
if 'pf_z' in train_df.columns:
    pf_z_resid = (train_df['pf_z'].values - train_df['last_known_tvt'].values).astype(np.float32)
    # mask out invalid pf_z rows (fallback zeros)
    valid_z = train_df['pf_z'].values != train_df['last_known_tvt'].values
    pf_blend = np.where(valid_z,
                        (pf_oof_resid + pf_z_resid) / 2.0,
                        pf_oof_resid).astype(np.float32)
    print('PF heuristic: 0.5*pf_ancc + 0.5*pf_z (residual space)')
else:
    pf_blend = pf_oof_resid
    print('PF heuristic: pf_ancc only (residual space)')

pf_oof = pf_blend
print(f'PF OOF RMSE (residual): {rmse(y_eval, pf_oof):.5f}')

# Test PF
if HAS_TEST and test_df is not None and 'pf_ancc' in test_df.columns:
    pf_test_resid = (test_df['pf_ancc'].values - test_df['last_known_tvt'].values).astype(np.float32)
    if 'pf_z' in test_df.columns:
        pf_z_t = (test_df['pf_z'].values - test_df['last_known_tvt'].values).astype(np.float32)
        valid_zt = test_df['pf_z'].values != test_df['last_known_tvt'].values
        pf_test = np.where(valid_zt, (pf_test_resid + pf_z_t) / 2.0, pf_test_resid).astype(np.float32)
    else:
        pf_test = pf_test_resid
else:
    pf_test = np.zeros(X_test.shape[0], dtype=np.float32) if HAS_TEST else np.array([], dtype=np.float32)
    if HAS_TEST:
        print('WARNING: pf_ancc not in test_df â€” pf_test = zeros')

PF heuristic: 0.5*pf_ancc + 0.5*pf_z (residual space)
PF OOF RMSE (residual): 13.74424


In [8]:
# -- Ablation A / B / C / D ----------------------------------------
ablation = {
    'A_heuristic_only':  rmse(y_eval, pf_oof),
    'B_ridge_stack_only': rmse(y_eval, oof_stack),
    'C_blend_030_070':   rmse(y_eval, stacker.blend(oof_stack, pf_oof)),
}
best_alpha, best_d = 0.30, float('inf')
for a in np.arange(0.0, 1.01, 0.05):
    r = rmse(y_eval, a * oof_stack + (1 - a) * pf_oof)
    if r < best_d: best_d, best_alpha = r, a
ablation['D_tuned_blend'] = best_d
ablation['D_best_alpha']  = round(float(best_alpha), 2)

print('=== EXP-007 Ablation (residual RMSE) ===')
for k, v in ablation.items():
    print(f'  {k}: {v:.5f}' if isinstance(v, float) else f'  {k}: {v}')
print(f'  Baseline (aw pipeline): 10.45212  LB 9.964')

=== EXP-007 Ablation (residual RMSE) ===
  A_heuristic_only: 13.74424
  B_ridge_stack_only: 10.56195
  C_blend_030_070: 12.14295
  D_tuned_blend: 10.54963
  D_best_alpha: 0.95000
  Baseline (aw pipeline): 10.45212  LB 9.964


In [9]:
# -- Final blend + save --------------------------------------------
final_alpha  = best_alpha
final_oof    = final_alpha * oof_stack  + (1 - final_alpha) * pf_oof
print(f'Final OOF RMSE (residual, alpha={final_alpha:.2f}): {rmse(y_eval, final_oof):.5f}')

# Save OOF (residual space â€” consistent with rogii-train-lgbm/catboost pipeline)
np.save(WORK_DIR / f'{EXP_ID}_final_oof.npy', final_oof)

if HAS_TEST and test_stack is not None:
    final_test = final_alpha * test_stack + (1 - final_alpha) * pf_test
    np.save(WORK_DIR / f'{EXP_ID}_final_test.npy', final_test)
    print(f'Final test predictions saved: {final_test.shape}')
else:
    print('HAS_TEST=False or no test_stack â€” test predictions skipped')

result = {
    'exp_id': EXP_ID, 'baseline_rmse': 10.45212,
    'has_test': HAS_TEST,
    'ablation': {k: round(float(v), 6) if isinstance(v, float) else v for k, v in ablation.items()},
    'final_rmse': round(rmse(y_eval, final_oof), 6),
    'blend_alpha': final_alpha, 'model_scores': model_scores,
    'tiny_sample': TINY_SAMPLE,
    'io_note': (
        'OOF in RESIDUAL space (TVT - last_known_tvt). '
        'Pass through EXP-009 projection postprocess next. '
        'For submission, use rogii-submission.ipynb with these weights.'
    )
}
with open(WORK_DIR / f'{EXP_ID}_result.json', 'w') as f:
    json.dump(result, f, indent=2)
print(json.dumps(result, indent=2))
print('\nNext: run exp009_projection_postprocess__CPU.ipynb on top of final_oof/final_test')

Final OOF RMSE (residual, alpha=0.95): 10.54963
HAS_TEST=False or no test_stack â€” test predictions skipped
{
  "exp_id": "EXP-007",
  "baseline_rmse": 10.45212,
  "has_test": false,
  "ablation": {
    "A_heuristic_only": 13.744237,
    "B_ridge_stack_only": 10.561951,
    "C_blend_030_070": 12.142951,
    "D_tuned_blend": 10.549633,
    "D_best_alpha": 0.95
  },
  "final_rmse": 10.549633,
  "blend_alpha": 0.9500000000000001,
  "model_scores": [
    {
      "model": "LGB-1",
      "cv_rmse": 10.714857
    },
    {
      "model": "LGB-2",
      "cv_rmse": 10.631486
    },
    {
      "model": "LGB-3",
      "cv_rmse": 10.598523
    },
    {
      "model": "CB-1",
      "cv_rmse": 10.669394
    },
    {
      "model": "CB-2",
      "cv_rmse": 10.697871
    },
    {
      "model": "CB-3",
      "cv_rmse": 10.673755
    }
  ],
  "tiny_sample": false,
  "io_note": "OOF in RESIDUAL space (TVT - last_known_tvt). Pass through EXP-009 projection postprocess next. For submission, use rogii-sub